# Fish Audio S2 Pro — Kaggle GPU Server for EdgeTTS-Studio

This notebook starts a **Fish Audio S2 Pro** TTS server on Kaggle's **T4 GPU** and exposes it to your PC via a **Cloudflare quick tunnel** (`https://xxxx.trycloudflare.com`).

## Kaggle settings (required)
1. **Settings → Accelerator → GPU T4 x2** — **not** P100, not CPU. Fish S2 + PyTorch 2.10 require CUDA compute **7.0+** (T4 = 7.5). P100 (6.0) will crash with `no kernel image is available for execution on the device`.
2. **Settings → Internet → On** (needed for `git clone`, pip, and cloudflared — **not** for model weights; see below)
3. After changing accelerator: **Save Version** (or Save), then **Run All**. If Kaggle still assigns P100, stop the session and try again later.
4. Keep this session **running** while you use EdgeTTS-Studio on your PC.

## Model weights — use a Kaggle Dataset (required)
**Do not download `fishaudio/s2-pro` from Hugging Face inside this notebook.** When Kaggle Internet is enabled, its internal proxy rewrites Hugging Face asset URLs and strips the `https://` scheme. That triggers `MissingSchema` errors and breaks `huggingface_hub` downloads.

Instead, stage the weights once on your PC and mount them as a dataset:

1. **Download locally** (~9 GB) from [fishaudio/s2-pro](https://huggingface.co/fishaudio/s2-pro) on your computer (Hugging Face CLI, `git lfs clone`, or the HF website).
2. **Create a Kaggle Dataset** — upload the checkpoint files (`codec.pth`, `model-*.safetensors`, `config.json`, tokenizer files, etc.). Private datasets work fine.
3. **Add the dataset to this notebook** — Notebook → **Add Data** → your dataset. It mounts read-only under `/kaggle/input/<dataset-slug>/`.
4. **Run All** — the checkpoint cell copies weights from the mounted dataset into `/kaggle/working/checkpoints/s2-pro`.
5. If auto-detect fails, set `FISH_S2_DATASET_SLUG` in the config cell (e.g. `"yourusername/fishaudio-s2-pro"`).

> **Fine-tuning note:** If you adapt this notebook for fine-tuning, upstream `config.json` / `tokenizer_config.json` may need tweaks for your tokenizer. This notebook targets **inference only**.

## About install warnings
- `Skipping acquire of configured file 'main/source/Sources'` — harmless Kaggle apt mirror notice.
- `libjack-jackd2-0 dependency problems` — normal when swapping audio libs; safe to ignore.
- `ldconfig ... is not a symbolic link` — harmless on Kaggle GPU images.
- Long `pip dependency conflicts` lists — caused by the old install pulling in torch/protobuf. **Restart the session** (Session → Restart Session), then re-run all cells. The updated install cell avoids that.
- `torchvision ... partially initialized module` / `Server exited early with code 1` — torch was downgraded to 2.8 while Kaggle torchvision needs 2.10. **Restart session**, re-run **all** cells; the install cell now repairs the torch stack automatically.
- `Cannot install descript-audio-codec` / `ResolutionImpossible` — old install tried to resolve conflicting protobuf pins. Re-run the install cell from the updated notebook (descript packages install with `--no-deps`).
- `Tesla P100` / `no kernel image is available for execution on the device` — Kaggle gave you a P100 instead of T4. Change accelerator to **GPU T4 x2**, restart session, run again.
- `CUDA out of memory` when loading `codec.pth` — S2 LLAMA fills one T4; the updated server puts the decoder on **cuda:1** (T4 x2) or **CPU** (single T4). Re-run the **write server** cell and **tunnel** cell.
- `index is on cuda:1, different from other tensors on cuda:0` — fish-speech upstream sends LLAMA tensors to the decoder GPU. Re-run the **write server** cell (uses `EdgeTTSInferenceEngine` fix) and the **tunnel** cell.
- `MissingSchema` / `Invalid URL` when downloading from Hugging Face — Kaggle's HF proxy rewrite. **Do not use `snapshot_download` here.** Upload weights as a Kaggle Dataset and re-run the checkpoint cell.
- `Fish S2 Pro weights not found` — add your dataset via **Add Data**, or set `FISH_S2_DATASET_SLUG` in the config cell.

## PC setup (EdgeTTS-Studio Native.exe)
1. Open **Remote GPU (Kaggle)** panel
2. Check **Fish Audio S2 server**
3. Paste the tunnel URL printed below (no trailing slash)
4. Select provider **Fish Audio S2 (Kaggle)** and synthesize

Click **Stop** in the app (or run the shutdown cell) when finished to free the GPU session.

In [ ]:
import os
import subprocess
import sys

WORK = "/kaggle/working"
FISH_ROOT = f"{WORK}/fish-speech"
CHECKPOINT = f"{WORK}/checkpoints/s2-pro"
SERVER_PORT = 7860

# Kaggle Dataset slug for fishaudio/s2-pro weights (set if auto-detect fails).
# Example: "yourusername/fishaudio-s2-pro" mounts at /kaggle/input/fishaudio-s2-pro/
# Leave empty to search all mounted /kaggle/input datasets for codec.pth.
FISH_S2_DATASET_SLUG = os.environ.get("FISH_S2_DATASET", "")
KAGGLE_INPUT = "/kaggle/input"

try:
    import torch
except ImportError:
    raise RuntimeError("PyTorch is missing. Use a Kaggle GPU image with PyTorch preinstalled.")

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected. In Kaggle: Settings → Accelerator → GPU T4 x2, then Save & Run All."
    )

props = torch.cuda.get_device_properties(0)
gpu_name = props.name
vram_gb = props.total_memory / (1024 ** 3)
major, minor = torch.cuda.get_device_capability(0)

print(f"GPU: {gpu_name} ({vram_gb:.1f} GB VRAM, compute {major}.{minor})")
print(f"Working dir: {WORK}")

_UNSUPPORTED = ("P100", "K80", "M60", "M40")
if major < 7 or any(tag in gpu_name.upper() for tag in _UNSUPPORTED):
    raise RuntimeError(
        f"Unsupported GPU: {gpu_name} (CUDA compute {major}.{minor}).\n"
        "Fish Audio S2 on Kaggle requires a T4 (compute 7.5) or newer.\n\n"
        "Fix:\n"
        "  1. Notebook Settings (right panel) → Accelerator → GPU T4 x2\n"
        "  2. Save, then Session → Restart Session\n"
        "  3. Run All again\n\n"
        "If you already selected T4 but still get P100, Kaggle may be out of T4 quota — "
        "try again in a few hours or on a different day."
    )

gpu_count = torch.cuda.device_count()
print(f"CUDA devices visible to this session: {gpu_count}")
if gpu_count < 2:
    print(
        "NOTE: Fish S2 LLAMA uses ~15 GB VRAM. With one T4, the codec decoder runs on CPU "
        "(slower reference-audio encode, but synthesis works). "
        "For best speed, use Settings → Accelerator → GPU T4 x2."
    )
else:
    print("T4 x2 detected — LLAMA will use cuda:0, codec decoder will use cuda:1.")

if "T4" not in gpu_name:
    print(f"WARNING: expected Tesla T4, got {gpu_name}. Continuing because compute {major}.{minor} >= 7.0.")

In [ ]:
# Install system + Python dependencies
# IMPORTANT: fish-speech's full pip install downgrades protobuf/torch and breaks Kaggle.
# We install fish-speech with --no-deps and only add inference-server packages.

def run(cmd: str) -> None:
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=True)


# Apt warnings (r2u mirror, libjack swap, ldconfig) are harmless on Kaggle.
run("apt-get update -qq")
run("DEBIAN_FRONTEND=noninteractive apt-get install -y -qq ffmpeg git wget libsox-dev")

if not os.path.isdir(FISH_ROOT):
    run(f"git clone --depth 1 https://github.com/fishaudio/fish-speech.git {FISH_ROOT}")

PY = sys.executable

# A prior fish-speech pip install may leave torch 2.8 while torchvision stays 0.25
# (requires torch 2.10) — that crashes the server subprocess. Repair if needed.
import torch

print(f"torch before repair: {torch.__version__}")


def _torch_stack_ok() -> bool:
    try:
        import torchvision
    except Exception:
        return False
    return torch.__version__.startswith("2.10") and torchvision.__version__.startswith("0.25")


if not _torch_stack_ok():
    print("Repairing torch/torchvision/torchaudio to Kaggle-compatible 2.10 stack …")
    run(
        f'{PY} -m pip install -q --force-reinstall "torch==2.10.0" '
        f'"torchvision==0.25.0" "torchaudio==2.10.0" '
        f"--index-url https://download.pytorch.org/whl/cu128"
    )
    import importlib

    importlib.invalidate_caches()
    importlib.reload(torch)
    import torchvision
    importlib.reload(torchvision)

    print(f"torch after repair: {torch.__version__}, torchvision {torchvision.__version__}")
else:
    import torchvision

    print(f"torch stack OK: {torch.__version__}, torchvision {torchvision.__version__}")

# Keep Kaggle's CUDA torch; block protobuf downgrade from descript-audio-codec.
constraints = f"{WORK}/pip-constraints.txt"
with open(constraints, "w", encoding="utf-8") as f:
    f.write("protobuf>=4.25.8,<6.0.0\n")

runtime_pkgs = [
    "fastapi",
    "uvicorn[standard]",
    "kui>=1.6.0",
    "loguru>=0.6.0",
    "pyrootutils>=1.0.4",
    "ormsgpack",
    "soundfile",
    "transformers<=4.57.3",
    "einops>=0.7.0",
    "einx[torch]==0.2.2",
    "librosa>=0.10.1",
    "resampy>=0.4.3",
    "safetensors",
    "tiktoken>=0.8.0",
    "zstandard>=0.22.0",
    "natsort>=8.4.0",
    "cachetools",
    "hydra-core>=1.3.2",
    "lightning>=2.1.0",
    "loralib>=0.1.2",
    "opencc-python-reimplemented==0.1.7",
    "silero-vad",
]

# descript-audio-codec pulls descript-audiotools which pins protobuf<3.20 — incompatible
# with Kaggle. Install it separately with --no-deps (fish-speech works with protobuf 4.x).
descript_deps = [
    "argbind>=0.3.7",
    "pyloudnorm",
    "importlib-resources",
    "julius",
    "ffmpy",
    "flatten-dict",
    "pystoi",
    "torch-stoi",
    "randomname",
]

run(f'PIP_CONSTRAINT="{constraints}" {PY} -m pip install -q -U "protobuf>=4.25.8,<6.0.0"')
run(f"{PY} -m pip install -q -e {FISH_ROOT} --no-deps")
run(f"{PY} -m pip install -q " + " ".join(f'"{p}"' for p in runtime_pkgs))
run(f"{PY} -m pip install -q " + " ".join(f'"{p}"' for p in descript_deps))
run(f"{PY} -m pip install -q descript-audiotools --no-deps")
run(f"{PY} -m pip install -q descript-audio-codec --no-deps")

# Quick import check before the heavy model download.
sys.path.insert(0, FISH_ROOT)
import torch

import torchvision

print(
    f"torch {torch.__version__}, torchvision {torchvision.__version__}, "
    f"cuda={torch.cuda.is_available()}"
)
from fish_speech.utils.schema import ServeTTSRequest  # noqa: F401
from fish_speech.inference_engine import TTSInferenceEngine  # noqa: F401
from fish_speech.models.dac.inference import load_model as load_decoder_model  # noqa: F401
import dac  # noqa: F401
import audiotools  # noqa: F401

import google.protobuf

print(f"protobuf {google.protobuf.__version__}")
print("Python deps OK (conflict warnings above, if any, are usually safe to ignore).")

# cloudflared binary for the public tunnel URL
cloudflared = f"{WORK}/cloudflared"
if not os.path.isfile(cloudflared):
    run(
        f"wget -q -O {cloudflared} https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x {cloudflared}"
    )

print("Dependencies ready.")

In [ ]:
# Mount Fish Audio S2 Pro weights from a Kaggle Dataset (~9 GB).
# Direct Hugging Face downloads fail on Kaggle when Internet is On (proxy strips https://).

import shutil
from pathlib import Path

CHECKPOINT_PATH = Path(CHECKPOINT)
KAGGLE_INPUT_PATH = Path(KAGGLE_INPUT)
MARKER = "codec.pth"

REQUIRED_WEIGHT_FILES = [
    "codec.pth",
    "model-00001-of-00002.safetensors",
    "model-00002-of-00002.safetensors",
    "model.safetensors.index.json",
    "config.json",
    "tokenizer.json",
    "tokenizer_config.json",
]

OPTIONAL_WEIGHT_FILES = [
    "chat_template.jinja",
    "special_tokens_map.json",
]


def _checkpoint_complete(path: Path) -> bool:
    return all((path / name).is_file() for name in REQUIRED_WEIGHT_FILES)


def _find_mounted_checkpoint() -> Path | None:
    if not KAGGLE_INPUT_PATH.is_dir():
        return None

    candidates: list[Path] = []

    if FISH_S2_DATASET_SLUG:
        slug = FISH_S2_DATASET_SLUG.strip().split("/")[-1]
        candidates.extend(
            [
                KAGGLE_INPUT_PATH / slug,
                KAGGLE_INPUT_PATH / slug / "s2-pro",
                KAGGLE_INPUT_PATH / slug / "fishaudio-s2-pro",
            ]
        )

    for marker in sorted(KAGGLE_INPUT_PATH.rglob(MARKER)):
        candidates.append(marker.parent)

    seen: set[Path] = set()
    for candidate in candidates:
        resolved = candidate.resolve()
        if resolved in seen:
            continue
        seen.add(resolved)
        if _checkpoint_complete(resolved):
            return resolved
    return None


def _copy_checkpoint(src: Path, dst: Path) -> None:
    dst.mkdir(parents=True, exist_ok=True)
    for name in REQUIRED_WEIGHT_FILES + OPTIONAL_WEIGHT_FILES:
        source = src / name
        if not source.is_file():
            if name in REQUIRED_WEIGHT_FILES:
                raise FileNotFoundError(f"Missing required file {name} in dataset at {src}")
            continue
        target = dst / name
        if target.is_file() and target.stat().st_size == source.stat().st_size:
            print(f"  skip {name} (already present)")
            continue
        print(f"  copy {name} …")
        shutil.copy2(source, target)


if _checkpoint_complete(CHECKPOINT_PATH):
    print("Model checkpoint already present in working dir — skipping copy.")
elif (mounted := _find_mounted_checkpoint()) is not None:
    print(f"Found mounted dataset at {mounted}")
    print(f"Copying weights to {CHECKPOINT_PATH} …")
    _copy_checkpoint(mounted, CHECKPOINT_PATH)
else:
    mounted_names = (
        ", ".join(sorted(p.name for p in KAGGLE_INPUT_PATH.iterdir() if p.is_dir()))
        if KAGGLE_INPUT_PATH.is_dir()
        else "(not on Kaggle — /kaggle/input missing)"
    )
    raise RuntimeError(
        "Fish S2 Pro weights not found.\n\n"
        "Kaggle's proxy breaks direct Hugging Face downloads (MissingSchema). "
        "Upload the fishaudio/s2-pro files as a Kaggle Dataset instead:\n"
        "  1. Download https://huggingface.co/fishaudio/s2-pro on your PC\n"
        "  2. Create a Kaggle Dataset and upload the checkpoint files\n"
        "  3. Notebook → Add Data → your dataset\n"
        "  4. Set FISH_S2_DATASET_SLUG in the config cell if auto-detect fails\n"
        "  5. Re-run from the config cell\n\n"
        f"Mounted datasets under {KAGGLE_INPUT}: {mounted_names or '(none)'}"
    )

if not _checkpoint_complete(CHECKPOINT_PATH):
    missing = [name for name in REQUIRED_WEIGHT_FILES if not (CHECKPOINT_PATH / name).is_file()]
    raise RuntimeError(f"Checkpoint incomplete at {CHECKPOINT_PATH}. Missing: {', '.join(missing)}")

print(f"Checkpoint ready: {CHECKPOINT}")

In [ ]:
# EdgeTTS-Studio compatibility server
# Matches RemoteFishSpeechEngine.cpp endpoints on your PC exe.
server_path = f"{WORK}/edgetts_fish_server.py"
server_code = r'''import base64
import io
import os
import queue
import sys
import threading
from typing import Any

import numpy as np
import soundfile as sf
import torch
import torchvision  # must load before fish_speech (torchvision/torch version match)
import uvicorn
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse, Response
from loguru import logger

if not (torch.__version__.startswith("2.10") and torchvision.__version__.startswith("0.25")):
    raise SystemExit(
        f"Incompatible torch stack in server: torch {torch.__version__}, "
        f"torchvision {torchvision.__version__}. Re-run the install cell."
    )

FISH_ROOT = os.environ["FISH_ROOT"]
CHECKPOINT = os.environ["CHECKPOINT"]
PORT = int(os.environ.get("SERVER_PORT", "7860"))

sys.path.insert(0, FISH_ROOT)
os.chdir(FISH_ROOT)

from fish_speech.inference_engine import TTSInferenceEngine
from fish_speech.models.dac.inference import load_model as load_decoder_model
from fish_speech.models.text2semantic.inference import (
    GenerateRequest,
    launch_thread_safe_queue,
)
from fish_speech.utils.schema import ServeTTSRequest
from tools.server.inference import inference_wrapper as inference
from tools.server.model_utils import cached_vqgan_batch_encode

USE_HALF = os.environ.get("USE_HALF", "1") == "1"
PRECISION = torch.half if USE_HALF else torch.bfloat16


def _pick_devices() -> tuple[str, str]:
    """S2-pro LLAMA alone fills a 16 GB T4; put the codec decoder on GPU 1 or CPU."""
    if not torch.cuda.is_available():
        return "cpu", "cpu"
    if torch.cuda.device_count() >= 2:
        return "cuda:0", "cuda:1"
    return "cuda", "cpu"


LLAMA_DEVICE, DECODER_DEVICE = _pick_devices()


class EdgeTTSInferenceEngine(TTSInferenceEngine):
    """fish-speech passes decoder_model.device to LLAMA; fix split-GPU placement."""

    def __init__(self, *args, llama_device: str, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        self.llama_device = torch.device(llama_device)

    def send_Llama_request(self, req, prompt_tokens, prompt_texts):
        request = dict(
            device=self.llama_device,
            max_new_tokens=req.max_new_tokens,
            text=req.text,
            top_p=req.top_p,
            repetition_penalty=req.repetition_penalty,
            temperature=req.temperature,
            compile=self.compile,
            iterative_prompt=req.chunk_length > 0,
            chunk_length=req.chunk_length,
            prompt_tokens=prompt_tokens,
            prompt_text=prompt_texts,
        )
        response_queue = queue.Queue()
        self.llama_queue.put(
            GenerateRequest(request=request, response_queue=response_queue)
        )
        return response_queue

    def decode_vq_tokens(self, codes):
        dec_dev = self.decoder_model.device
        if codes.device != dec_dev:
            codes = codes.to(dec_dev)
        return super().decode_vq_tokens(codes)


app = FastAPI(title="EdgeTTS-Studio Fish Audio S2 Server", version="1.0")
engine: EdgeTTSInferenceEngine | None = None
shutdown_event = threading.Event()


def _sample_rate() -> int:
    dec = engine.decoder_model
    if hasattr(dec, "spec_transform"):
        return dec.spec_transform.sample_rate
    return dec.sample_rate


def _audio_to_wav_bytes(audio: np.ndarray, fmt: str = "wav") -> bytes:
    buf = io.BytesIO()
    sf.write(buf, audio, _sample_rate(), format=fmt)
    return buf.getvalue()


def _decode_reference_audio(ref: dict[str, Any]) -> bytes:
    audio = ref.get("audio", b"")
    if isinstance(audio, str):
        return base64.b64decode(audio)
    if isinstance(audio, (bytes, bytearray)):
        return bytes(audio)
    raise ValueError("Reference audio must be base64 string or bytes")


def _tts_with_codes(req: ServeTTSRequest, refs: list[dict[str, Any]]) -> bytes:
    prompt_tokens, prompt_texts = [], []
    for ref in refs:
        if "codes" in ref:
            codes = torch.tensor(ref["codes"], dtype=torch.long)
            prompt_tokens.append(codes)
        else:
            prompt_tokens.append(engine.encode_reference(_decode_reference_audio(ref), True))
        prompt_texts.append(ref.get("text", ""))

    response_queue = engine.send_Llama_request(req, prompt_tokens, prompt_texts)
    segments: list[np.ndarray] = []
    while True:
        wrapped = response_queue.get()
        if wrapped.status == "error":
            raise RuntimeError(str(wrapped.response))
        result = wrapped.response
        if getattr(result, "action", None) == "next":
            break
        segments.append(engine.get_audio_segment(result))

    if not segments:
        raise RuntimeError("No audio generated")
    return _audio_to_wav_bytes(np.concatenate(segments, axis=0), req.format)


@app.on_event("startup")
def _startup() -> None:
    global engine
    logger.info(
        "Loading Fish Audio S2 Pro — LLAMA on {}, decoder on {}, half={}",
        LLAMA_DEVICE,
        DECODER_DEVICE,
        USE_HALF,
    )

    # Load the smaller codec first on the secondary device so LLAMA can own GPU 0.
    decoder = load_decoder_model(
        "modded_dac_vq",
        os.path.join(CHECKPOINT, "codec.pth"),
        device=DECODER_DEVICE,
    )

    llama_queue = launch_thread_safe_queue(
        checkpoint_path=CHECKPOINT,
        device=LLAMA_DEVICE,
        precision=PRECISION,
        compile=False,
    )

    engine = EdgeTTSInferenceEngine(
        llama_queue=llama_queue,
        decoder_model=decoder,
        precision=PRECISION,
        compile=False,
        llama_device=LLAMA_DEVICE,
    )

    # Warm-up (first inference is slow; also validates the split-device setup).
    warm = ServeTTSRequest(
        text="Hello.",
        references=[],
        reference_id=None,
        max_new_tokens=128,
        chunk_length=200,
        top_p=0.7,
        repetition_penalty=1.2,
        temperature=0.7,
        format="wav",
    )
    list(inference(warm, engine))
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    logger.info("Model ready.")


@app.get("/")
async def root():
    return {"status": "ok", "service": "fish-audio-s2-pro", "client": "EdgeTTS-Studio"}


@app.get("/v1/health")
async def health():
    return {"status": "ok"}


@app.post("/v1/tts")
async def tts(request: Request):
    body = await request.json()
    allowed = set(ServeTTSRequest.model_fields.keys())
    req = ServeTTSRequest(**{k: v for k, v in body.items() if k in allowed})
    refs = body.get("references") or []

    try:
        if any(isinstance(r, dict) and "codes" in r for r in refs):
            wav = _tts_with_codes(req, refs)
        else:
            audio = next(inference(req, engine))
            wav = _audio_to_wav_bytes(audio, req.format)
        return Response(content=wav, media_type="audio/wav")
    except Exception as exc:
        logger.exception("TTS failed")
        return JSONResponse({"detail": str(exc)}, status_code=500)


@app.post("/v1/models/vqgan/encode")
async def vqgan_encode(request: Request):
    body = await request.json()
    audio_b64 = body.get("audio")
    if not audio_b64:
        return JSONResponse({"detail": "Missing 'audio' (base64 WAV/MP3 bytes)"}, status_code=400)
    try:
        audio_bytes = base64.b64decode(audio_b64)
        tokens = cached_vqgan_batch_encode(engine.decoder_model, [audio_bytes])
        codes = tokens[0].tolist()
        # Saved locally by EdgeTTS-Studio and sent back as a reference object.
        return JSONResponse({"codes": codes, "text": body.get("text", "")})
    except Exception as exc:
        logger.exception("VQGAN encode failed")
        return JSONResponse({"detail": str(exc)}, status_code=500)


@app.post("/shutdown")
async def shutdown():
    shutdown_event.set()
    threading.Thread(target=lambda: os._exit(0), daemon=True).start()
    return {"status": "shutting_down"}


if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="info")
'''

with open(server_path, "w", encoding="utf-8") as f:
    f.write(server_code)

print(f"Wrote {server_path}")

In [ ]:
# Start server + cloudflared tunnel, then print the URL for EdgeTTS-Studio
import re
import threading
import time

import torch
import torchvision

if not (torch.__version__.startswith("2.10") and torchvision.__version__.startswith("0.25")):
    raise RuntimeError(
        f"Incompatible torch stack: torch {torch.__version__}, torchvision {torchvision.__version__}. "
        "Re-run the install cell above first."
    )
print(f"Launching server with torch {torch.__version__}, torchvision {torchvision.__version__}")

env = os.environ.copy()
env["FISH_ROOT"] = FISH_ROOT
env["CHECKPOINT"] = CHECKPOINT
env["SERVER_PORT"] = str(SERVER_PORT)
env["USE_HALF"] = "1"  # FP16 — required for 16 GB T4 VRAM
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

server_proc = subprocess.Popen(
    [sys.executable, f"{WORK}/edgetts_fish_server.py"],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)


def _stream_server_logs():
    assert server_proc.stdout is not None
    for line in server_proc.stdout:
        print(line, end="")


threading.Thread(target=_stream_server_logs, daemon=True).start()

print("Waiting for Fish S2 server to load (first run: model warm-up may take 2–5 min)…")
for _ in range(600):
    if server_proc.poll() is not None:
        raise RuntimeError(f"Server exited early with code {server_proc.returncode}")
    try:
        import urllib.request

        with urllib.request.urlopen(f"http://127.0.0.1:{SERVER_PORT}/", timeout=2) as resp:
            if resp.status == 200:
                print("Server is up on localhost.")
                break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError("Server did not become ready within 20 minutes.")

tunnel_proc = subprocess.Popen(
    [f"{WORK}/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{SERVER_PORT}"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

PUBLIC_URL = None
url_pattern = re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")
deadline = time.time() + 120
while time.time() < deadline and PUBLIC_URL is None:
    line = tunnel_proc.stdout.readline() if tunnel_proc.stdout else ""
    if line:
        print(line, end="")
        match = url_pattern.search(line)
        if match:
            PUBLIC_URL = match.group(0)
    elif tunnel_proc.poll() is not None:
        break

if not PUBLIC_URL:
    raise RuntimeError("Could not read trycloudflare URL from cloudflared output.")

print("\n" + "=" * 72)
print("COPY THIS URL INTO EdgeTTS-Studio → Remote GPU → Fish Audio S2 server")
print(PUBLIC_URL)
print("=" * 72)
print("Keep this notebook running. First synthesis may be slow while CUDA warms up.")
print("When finished, click Stop in the app or interrupt this cell (Kernel → Interrupt).")

In [ ]:
# Keep the Kaggle session alive while your PC uses the tunnel.
# Interrupt this cell to end the session.
import time

while True:
    if server_proc.poll() is not None:
        raise RuntimeError(f"Server process stopped (exit {server_proc.returncode}). Re-run previous cells.")
    if tunnel_proc.poll() is not None:
        raise RuntimeError("Cloudflared tunnel stopped. Re-run the tunnel cell.")
    time.sleep(30)